In [23]:
"""
Digital Earth Australia Wetlands Insight Tool widget, which can be used to draw a polygon around an area of interest to extract a stacked line plot showing open water, wet, green, dry and brown vegetation percentages.
"""

# Import required packages
import fiona
import os
import sys
import datacube
import warnings
import matplotlib.pyplot as plt
from datacube.utils.geometry import CRS
from ipyleaflet import (
    WMSLayer,
    basemaps,
    basemap_to_tiles,
    Map,
    DrawControl,
    WidgetControl,
    SearchControl,
    Marker,
    LayerGroup,
    LayersControl,
    GeoData,
)
from traitlets import Unicode
from ipywidgets import (
    GridspecLayout,
    Button,
    Layout,
    HBox,
    VBox,
    HTML,
    Output,
)
import json
import itertools
import numpy as np
import geopandas as gpd
from io import BytesIO
import ipyleaflet as leaflet
import ipywidgets as widgets
import datetime
from skimage import exposure
from skimage.filters import unsharp_mask
import seaborn as sns
from shapely.geometry import box, shape

sys.path.insert(1, "../Tools/")
from datacube.utils import masking
from datacube.utils.geometry import Geometry
import dea_tools.app.widgetconstructors as deawidgets
from dea_tools.dask import create_local_dask_cluster
from dea_tools.spatial import reverse_geocode
from dea_tools.datahandling import xr_pansharpen
import dea_tools.wetlands
from dea_tools.wetlands import generate_low_quality_data_periods
from dea_tools.wit import WIT_drill

# Import required packages
import fiona
import os
import sys
import datacube
import warnings
import matplotlib.pyplot as plt
from datacube.utils.geometry import CRS
from ipyleaflet import (
    WMSLayer,
    basemaps,
    basemap_to_tiles,
    Map,
    DrawControl,
    WidgetControl,
    LayerGroup,
    LayersControl,
    GeoData,
)
from traitlets import Unicode
from ipywidgets import (
    GridspecLayout,
    Button,
    Layout,
    HBox,
    VBox,
    HTML,
    Output,
)
import json
import geopandas as gpd
from io import BytesIO
import ipywidgets as widgets

import dea_tools.app.widgetconstructors as deawidgets
from dea_tools.coastal import get_coastlines, transect_distances


In [24]:
def make_box_layout():
    return Layout(
        # border='solid 1px black',
        margin="0px 10px 10px 0px",
        padding="5px 5px 5px 5px",
        width="100%",
        height="100%",
    )

In [25]:
def create_expanded_button(description, button_style):
    return Button(
        description=description,
        button_style=button_style,
        layout=Layout(width="auto", height="auto"),
    )

In [26]:
class wit_app(HBox):
    
    def __init__(self):
        super().__init__()

        ######################
        # INITIAL ATTRIBUTES #
        ######################

        self.product_list = [
            ("ESRI World Imagery", "none"),
            ("Open Street Map", "open_street_map"),
        ]
        self.product = self.product_list[0][1]
        #self.product_year = "2024-01-01"
        self.target = None
        self.action = None
        self.gdf_drawn = None
        self.gdf_uploaded = None
        
        # Create the Header widget
        header_title_text = "<h3>Digital Earth Australia Wetlands Insight Tool</h3>"
        instruction_text = "Select parameters and draw a polygon on the map to extract a staked line plot for a given area."
        self.header = deawidgets.create_html(
            f"<h3>{header_title_text}</h3><p>{instruction_text}</p>"
        )
        self.header.layout = make_box_layout()

        #####################################
        # HANDLER FUNCTION FOR DRAW CONTROL #
        #####################################

        # Define the action to take once something is drawn on the map
        def update_geojson(target, action, geo_json):

            # Remove previously uploaded data if present
            self.gdf_uploaded = None
            fileupload_wetlands._counter = 0

            # Get data from action
            self.action = action

            # Convert data to geopandas

            json_data = json.dumps(geo_json)
            binary_data = json_data.encode()
            io = BytesIO(binary_data)
            io.seek(0)
            gdf = gpd.read_file(io)
            gdf.crs = "EPSG:4326"
            
           # Convert to Albers and compute area
            gdf_drawn_albers = gdf.copy().to_crs("EPSG:3577")
            m2_per_km2 = 10**6
            area = gdf_drawn_albers.envelope.area.values[0] / m2_per_km2
            polyarea_label = 'Total area of DEA Coastlines data to extract'
            polyarea_text = f"<b>{polyarea_label}</b>: {area:.2f} km<sup>2</sup>"
            
        ###########################
        # WIDGETS FOR APP OUTPUTS #
        ###########################

        self.dask_client = Output(layout=make_box_layout())

        self.wit_plot = Output(layout=make_box_layout())

        self.status_info = Output(layout=make_box_layout())
        #self.wit_plot = Output(layout=make_box_layout())

        #########################################
        # MAP WIDGET, DRAWING TOOLS, WMS LAYERS #
        #########################################

        # Create drawing tools
        desired_drawtools = ["rectangle", "polygon"]
        draw_control = deawidgets.create_drawcontrol(desired_drawtools)

        # Begin by displaying an empty layer group, and update the group with desired WMS on interaction.
        self.map_layers = LayerGroup(layers=())
        self.map_layers.name = "Map Overlays"

        # Create map widget
        self.m = deawidgets.create_map(map_center=(-28, 135),
                                       zoom_level=4,
                                       basemap=basemaps.Esri.WorldImagery)
        self.m.layout = make_box_layout()

        # Add tools to map widget
        self.m.add_control(draw_control)
       
        self.m.add_layer(self.map_layers)

        # Store current basemap for future use
        self.basemap = self.m.basemap

        ############################
        # WIDGETS FOR APP CONTROLS #
        ############################

        # Create parameter widgets

        
        run_button = create_expanded_button(("Run"), "info")
        
        fileupload_wetlands = widgets.FileUpload(accept='', multiple=True)

        ####################################
        # UPDATE FUNCTIONS FOR EACH WIDGET #
        ####################################


        run_button.on_click(self.run_app)
        draw_control.on_draw(update_geojson)
        fileupload_wetlands.observe(self.update_fileupload_wetlands, "value")
        
        ##################################
        # COLLECTION OF ALL APP CONTROLS #
        ##################################

        parameter_selection = VBox(
            [
                HTML(
                "</br><i><b>Advanced</b></br>Upload a GeoJSON or ESRI "
                "Shapefile (<5 mb) containing one or more wetland polygons.</i>"),
            fileupload_wetlands
            ]
        )

        
        parameter_selection.layout = make_box_layout()

        ###############################
        # SPECIFICATION OF APP LAYOUT #
        ###############################

        # Create the layout #[rowspan, colspan]
        grid = GridspecLayout(12, 10, height="1350px", width="auto")

        # Controls and Status
        grid[0, :8] = self.header

        grid[1:6, 0:2] = parameter_selection
        grid[6, 0:2] = run_button

        # Dask and Progress info
        grid[8:9, :] = self.status_info

        # Map
        grid[1:7, 2:] = self.m

        # Plot
        grid[9:, :] = self.wit_plot

        # Display using HBox children attribute
        self.children = [grid]

    ######################################
    # DEFINITION OF ALL UPDATE FUNCTIONS #
    ######################################

    # Set the output csv
    def update_fileupload_wetlands(self, change):
    
        # Clear any drawn data if present
        self.gdf_drawn = None
        
        # Temporary compatibility fix for ipywidget > 8.0
        # TODO: Update code to use new fileupload API documented here:
        # https://ipywidgets.readthedocs.io/en/latest/user_migration_guides.html#fileupload
        uploaded_data = {f["name"]: {"content": f.content.tobytes()} for f in change.new}            

        # Save to file
        for uploaded_filename in uploaded_data.keys():
            with open(uploaded_filename, "wb") as output_file:
                content = uploaded_data[uploaded_filename]["content"]
                output_file.write(content)

        with self.status_info:

            try:            

                print('Loading vector data...', end='\r')
                valid_files = [
                    file for file in uploaded_data.keys()
                    if file.lower().endswith(('.shp', '.geojson'))
                ]
                valid_file = valid_files#[0]
                wetlands_gdf = (gpd.read_file(valid_file).to_crs(
                    "EPSG:4326").explode(index_parts=True).reset_index(drop=True))

                # Use ID column if it exists
                if 'id' in wetlands_gdf:
                    wetlands_gdf = wetlands_gdf.set_index('id')
                    print(f"Uploaded '{valid_file}'; automatically labelling "
                          "transects using column 'id'.")
                else:
                    print(
                        f"Uploaded '{valid_file}'; no 'id' column detected, "
                        f"labelling transects from 0 to {len(wetlands_gdf.index) - 1}."
                    )

                # Create a geodata
                geodata = GeoData(geo_dataframe=wetlands_gdf,
                                  style={
                                      'color': 'black',
                                      'weight': 3
                                  })

                # Add to map
                xmin, ymin, xmax, ymax = wetlands_gdf.total_bounds
                self.m.fit_bounds([[ymin, xmin], [ymax, xmax]])
                self.m.add_layer(geodata)

                # If completed, add to attribute
                self.gdf_uploaded = wetlands_gdf

            except IndexError:
                print(
                    "Cannot read uploaded files. Please ensure that data is "
                    "in either GeoJSON or ESRI Shapefile format.",
                    end='\r')
                self.gdf_uploaded = None

            except fiona.errors.DriverError:
                print(
                    "Shapefile is invalid. Please ensure that all shapefile "
                    "components (e.g. .shp, .shx, .dbf, .prj) are uploaded.",
                    end='\r')
                self.gdf_uploaded = None
    



    def run_app(self, change):

        # Clear progress bar and output areas before running
        self.status_info.clear_output()
        self.wit_plot.clear_output()

        # run wetlands polygon drill
        with self.status_info:
            #             with ProgressBar():
            warnings.filterwarnings("ignore")

            # Load polygons from either map or uploaded files
            if self.gdf_uploaded is not None:
                wetlands_gdf = self.gdf_uploaded
                run_text = 'uploaded file'
            elif self.gdf_drawn is not None:
                wetlands_gdf = self.gdf_drawn
                #wetlands_gdf.index = [self.output_name]
                run_text = 'selected polygon'
            else:
                print(f'No transect drawn or uploaded. Please select a transect on the map, or upload a GeoJSON or ESRI Shapefile.',
                      end='\r')
                wetlands_gdf = None
  

In [27]:
wit_app()

wit_app(children=(GridspecLayout(children=(HTML(value='<h3><h3>Digital Earth Australia Wetlands Insight Tool</…

In [33]:
"""
Digital Earth Australia Coastline widget, which can be used to 
interactively extract shoreline data using transects.
"""

# Import required packages
import fiona
import os
import sys
import datacube
import warnings
import matplotlib.pyplot as plt
from datacube.utils.geometry import CRS
from ipyleaflet import (
    WMSLayer,
    basemaps,
    basemap_to_tiles,
    Map,
    DrawControl,
    WidgetControl,
    LayerGroup,
    LayersControl,
    GeoData,
)
from traitlets import Unicode
from ipywidgets import (
    GridspecLayout,
    Button,
    Layout,
    HBox,
    VBox,
    HTML,
    Output,
)
import json
import geopandas as gpd
from io import BytesIO
import ipywidgets as widgets

import dea_tools.app.widgetconstructors as deawidgets
from dea_tools.coastal import get_coastlines, transect_distances


WMS_ADDRESS = "https://geoserver.dea.ga.gov.au/geoserver/wms"


def make_box_layout():
    return Layout(
        #          border='solid 1px black',
        margin='0px 10px 10px 0px',
        padding='5px 5px 5px 5px',
        width='100%',
        height='100%',
    )


def create_expanded_button(description, button_style):
    return Button(
        description=description,
        button_style=button_style,
        layout=Layout(width="auto", height="auto"),
    )


class transect_app(HBox):

    def __init__(self):
        super().__init__()

        ######################
        # INITIAL ATTRIBUTES #
        ######################


        self.product_list = [
            ("ESRI World Imagery", "none"),
            ("Open Street Map", "open_street_map"),
        ]
        self.product = self.product_list[0][1]
        self.target = None
        self.action = None
        self.gdf_drawn = None
        self.gdf_uploaded = None

        ##################
        # HEADER FOR APP #
        ##################

        # Create the Header widget
        header_title_text = "<h3>Digital Earth Australia Coastlines shoreline transect extraction</h3>"
        instruction_text = "Select parameters and draw a transect on the map to extract shoreline data. <b>In distance mode</b>, draw a transect line starting from land that crosses multiple shorelines. <br><b>In width mode</b>, draw a transect line that intersects shorelines at least twice. Alternatively, <b>upload an vector file</b> to extract shoreline data for multiple existing transects."
        self.header = deawidgets.create_html(
            f"{header_title_text}<p>{instruction_text}</p>")
        self.header.layout = make_box_layout()

        #####################################
        # HANDLER FUNCTION FOR DRAW CONTROL #
        #####################################

        # Define the action to take once something is drawn on the map
        def update_geojson(target, action, geo_json):

            # Remove previously uploaded data if present
            self.gdf_uploaded = None
            fileupload_transects._counter = 0

            # Get data from action
            self.action = action

            # Convert data to geopandas
            json_data = json.dumps(geo_json)
            binary_data = json_data.encode()
            io = BytesIO(binary_data)
            io.seek(0)
            gdf = gpd.read_file(io)
            gdf.crs = "EPSG:4326"

            # Convert to Albers and compute area
            gdf_drawn_albers = gdf.copy().to_crs("EPSG:3577")
            m2_per_km2 = 10**6
            area = gdf_drawn_albers.envelope.area.values[0] / m2_per_km2


        ###########################
        # WIDGETS FOR APP OUTPUTS #
        ###########################

        self.status_info = Output(layout=make_box_layout())
        self.output_plot = Output(layout=make_box_layout())

        #########################################
        # MAP WIDGET, DRAWING TOOLS, WMS LAYERS #
        #########################################

        # Create drawing tools
        desired_drawtools = ['polyline']
        draw_control = deawidgets.create_drawcontrol(desired_drawtools)



        # Begin by displaying an empty layer group, and update the group with desired WMS on interaction.
        self.map_layers = LayerGroup()
        self.map_layers.name = 'Map Overlays'

        # Create map widget
        self.m = deawidgets.create_map(map_center=(-28, 135),
                                       zoom_level=4,
                                       basemap=basemaps.Esri.WorldImagery)
        self.m.layout = make_box_layout()

        # Add tools to map widget
        self.m.add_control(draw_control)
        self.m.add_layer(self.map_layers)

        # Store current basemap for future use
        self.basemap = self.m.basemap

        ############################
        # WIDGETS FOR APP CONTROLS #
        ############################

        run_button = create_expanded_button("Extract shoreline data", "info")
        fileupload_transects = widgets.FileUpload(accept='', multiple=True)

        ####################################
        # UPDATE FUNCTIONS FOR EACH WIDGET #
        ####################################

        # Run update functions whenever various widgets are changed.
        run_button.on_click(self.run_app)
        draw_control.on_draw(update_geojson)
        fileupload_transects.observe(self.update_fileupload_transects, "value")

        ##################################
        # COLLECTION OF ALL APP CONTROLS #
        ##################################

        parameter_selection = VBox([
            HTML(
                "</br><i><b>Advanced</b></br>Upload a GeoJSON or ESRI "
                "Shapefile (<5 mb) containing one or more transect lines.</i>"),
            fileupload_transects
        ])
       
        parameter_selection.layout = make_box_layout()  
        
        ###############################
        # SPECIFICATION OF APP LAYOUT #
        ###############################

        #       0   1    2   3   4   5   6   7    8   9
        #     ---------------------------------------------
        # 0   | Header                         | Map sel. |
        #     ---------------------------------------------
        # 1   | Params |                                  |
        # 2   |        |                                  |
        # 3   |        |                                  |
        # 4   |        |               Map                |
        # 5   |        |                                  |
        #     ----------                                  |
        # 6   |  Run   |                                  |
        #     ---------------------------------------------
        # 7   |               Status info                 |
        #     ---------------------------------------------
        # 8   |                                           |
        # 9   |               Output/figure               |
        # 10  |                                           |
        # 11  | ------------------------------------------|

        # Create the layout #[rowspan, colspan]
        grid = GridspecLayout(12, 10, height="1350px", width="auto")

        # Header and controls
        grid[0, :8] = self.header

        grid[1:6, 0:2] = parameter_selection
        grid[6, 0:2] = run_button

        # Status info, map and plot
        grid[1:7, 2:] = self.m  # map
        grid[7:8, :] = self.status_info
        grid[8:, :] = self.output_plot

        # Display using HBox children attribute
        self.children = [grid]

    ######################################
    # DEFINITION OF ALL UPDATE FUNCTIONS #
    ######################################

    # Set the output csv
    def update_fileupload_transects(self, change):

        # Clear any drawn data if present
        self.gdf_drawn = None
        
        # Temporary compatibility fix for ipywidget > 8.0
        # TODO: Update code to use new fileupload API documented here:
        # https://ipywidgets.readthedocs.io/en/latest/user_migration_guides.html#fileupload
        uploaded_data = {f["name"]: {"content": f.content.tobytes()} for f in change.new}            

        # Save to file
        for uploaded_filename in uploaded_data.keys():
            with open(uploaded_filename, "wb") as output_file:
                content = uploaded_data[uploaded_filename]["content"]
                output_file.write(content)

        with self.status_info:

            try:            

                print('Loading vector data...', end='\r')
                valid_files = [
                    file for file in uploaded_data.keys()
                    if file.lower().endswith(('.shp', '.geojson'))
                ]
                valid_file = valid_files[0]
                transect_gdf = (gpd.read_file(valid_file).to_crs(
                    "EPSG:4326").explode(index_parts=True).reset_index(drop=True))

                # Use ID column if it exists
                if 'id' in transect_gdf:
                    transect_gdf = transect_gdf.set_index('id')
                    print(f"Uploaded '{valid_file}'; automatically labelling "
                          "transects using column 'id'.")
                else:
                    print(
                        f"Uploaded '{valid_file}'; no 'id' column detected, "
                        f"labelling transects from 0 to {len(transect_gdf.index) - 1}."
                    )

                # Create a geodata
                geodata = GeoData(geo_dataframe=transect_gdf,
                                  style={
                                      'color': 'black',
                                      'weight': 3
                                  })

                # Add to map
                xmin, ymin, xmax, ymax = transect_gdf.total_bounds
                self.m.fit_bounds([[ymin, xmin], [ymax, xmax]])
                self.m.add_layer(geodata)

                # If completed, add to attribute
                self.gdf_uploaded = transect_gdf

            except IndexError:
                print(
                    "Cannot read uploaded files. Please ensure that data is "
                    "in either GeoJSON or ESRI Shapefile format.",
                    end='\r')
                self.gdf_uploaded = None

            except fiona.errors.DriverError:
                print(
                    "Shapefile is invalid. Please ensure that all shapefile "
                    "components (e.g. .shp, .shx, .dbf, .prj) are uploaded.",
                    end='\r')
                self.gdf_uploaded = None

    def run_app(self, change):

        # Clear progress bar and output areas before running
        self.status_info.clear_output()
        self.output_plot.clear_output()

        # Run DEA Coastlines analysis
        with self.status_info:
            warnings.filterwarnings("ignore")

            # Load transects from either map or uploaded files
            if self.gdf_uploaded is not None:
                transect_gdf = self.gdf_uploaded
                run_text = 'uploaded file'
            elif self.gdf_drawn is not None:
                transect_gdf = self.gdf_drawn
                transect_gdf.index = [self.output_name]
                run_text = 'selected transect'
            else:
                print(f'No transect drawn or uploaded. Please select a transect on the map, or upload a GeoJSON or ESRI Shapefile.',
                      end='\r')
                transect_gdf = None



In [34]:
transect_app()


transect_app(children=(GridspecLayout(children=(HTML(value='<h3>Digital Earth Australia Coastlines shoreline t…